## **🧩ربط Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **🧩 استيراد المكتبات**

In [2]:
import pandas as pd
import numpy as np
import os
import glob

# **🧩  قراءة أحدث ملف**

In [3]:
folder_path = "/content/drive/MyDrive/ملفات الهلال الاحمر من ديسمبر لين يناير/"
files = glob.glob(folder_path + "*.xlsx")

latest_file = max(files, key=os.path.getctime)
print(f"✅ جاري قراءة الملف: {latest_file}")

df = pd.read_excel(latest_file)

✅ جاري قراءة الملف: /content/drive/MyDrive/ملفات الهلال الاحمر من ديسمبر لين يناير/4-10 يناير.xlsx


# **🧩 تنظيف البيانات**

In [4]:
df.columns = df.columns.str.strip()

numeric_cols = [
    'القيمة الاقتصادية',
    'حالات غير منقولة',
    'حالات منقولة',
    'عدد المستفيدين',
    'عدد المتطوعات',
    'عدد المتطوعين'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'مدينة / محافظة' in df.columns:
    df['مدينة / محافظة'] = df['مدينة / محافظة'].str.strip()

print("✅ تم تنظيف البيانات")

✅ تم تنظيف البيانات


# **🧩إنشاء الأعمدة**

In [5]:
df['إجمالي المتطوعين'] = df['عدد المتطوعين'] + df['عدد المتطوعات']

if 'Total Hours Accepted' in df.columns:
    df['إجمالي الساعات'] = pd.to_numeric(df['Total Hours Accepted'], errors='coerce').fillna(0)
else:
    df['إجمالي الساعات'] = pd.to_numeric(df.get('مجموع الساعات', 0), errors='coerce').fillna(0)

# **🧩 التجميع (GroupBy)**

In [6]:
weekly_report = df.groupby('مدينة / محافظة').agg({
    'اسم الفرصة التطوعية': 'count',
    'عدد المتطوعات': 'sum',
    'عدد المتطوعين': 'sum',
    'إجمالي المتطوعين': 'sum',
    'إجمالي الساعات': 'sum',
    'عدد المستفيدين': 'sum',
    'حالات منقولة': 'sum',
    'حالات غير منقولة': 'sum',
    'القيمة الاقتصادية': 'sum'
}).reset_index()

weekly_report.columns = [
    'City',
    'Opportunities Count',
    'Female Volunteers',
    'Male Volunteers',
    'Overall Total Volunteers',
    'Total Volunteering Hours',
    'Total Beneficiaries',
    'Transported Cases',
    'Non-Transported Cases',
    'Economic Value'
]

display(weekly_report)

,City,Opportunities Count,Female Volunteers,Male Volunteers,Overall Total Volunteers,Total Volunteering Hours,Total Beneficiaries,Transported Cases,Non-Transported Cases,Economic Value
0,الطائف,10,40,46,86,744,750,3,393,39576
1,القنفذة,1,0,5,5,48,1,1,0,2728
2,جدة,24,25,82,107,653,270,1,32,36809
3,رابغ,2,0,2,2,48,20,0,0,2208
4,مكة المكرمة,13,106,177,283,2462,254,10,118,139579


In [11]:
from google.colab import auth
import gspread
from google.auth import default

# 1) تسجيل الدخول
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2) اسم الشيت
spreadsheet_name = "Makkah_Weekly_Report_Final"

try:
    # فتح الشيت
    sh = gc.open(spreadsheet_name)
    worksheet = sh.get_worksheet(0)

    # 3) تجهيز البيانات

     # تجهيز البيانات
    df_to_upload = weekly_report.astype(str)

    data_to_upload = [df_to_upload.columns.tolist()] + df_to_upload.values.tolist()

    # 4) تحديث البيانات
    worksheet.clear()
    worksheet.update(data_to_upload)

    print("🚀 تم تحديث Google Sheets بنجاح!")

except Exception as e:
    print(f"❌ خطأ أثناء التحديث: {e}")

🚀 تم تحديث Google Sheets بنجاح!
